# Lesson 4.5: The Bootstrap (Bonus)

Bootstrap one small sample of five heart rates: get a standard error and a 95% interval without any formula, then compare with the t-interval.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A researcher has only five resting heart rates (62, 70, 68, 75, 65 bpm). How uncertain is their mean, and can we answer without trusting a formula or the assumption of normal data? We will resample the five values thousands of times (the bootstrap) and compare the result with the classic t-interval.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: The sample

Five resting heart rates (bpm) from the lesson. The classic formula SE = s / sqrt(n) is our benchmark.

In [1]:
import numpy as np  # Load NumPy for arrays and arithmetic.
from scipy import stats  # Load SciPy's stats tools, which we need for the t-interval.

data = np.array([62, 70, 68, 75, 65])  # Store the five heart rates in an array.
n = len(data)  # Count them: n = 5 is a very small sample.
se = data.std(ddof=1) / np.sqrt(n)  # Classic standard error: sample SD (ddof=1) divided by the square root of n.

print(f'mean = {data.mean():.1f}')  # Print the sample mean.
print(f's    = {data.std(ddof=1):.2f}')  # Print the sample SD.
print(f'classic SE = {se:.2f}')  # Print the classic standard error, our benchmark.

mean = 68.0
s    = 4.95
classic SE = 2.21


**What this means.** The mean heart rate is 68.0 bpm, the SD is 4.95 and the classic standard error is 2.21. So the formula says the true average is known to within about 2 bpm. Next we get this uncertainty without the formula.

## Step 2: Resample 5,000 times

Each resample draws 5 values from our data with replacement (the same person can appear twice), then takes the mean. We keep all 5,000 means.

In [2]:
rng = np.random.default_rng(2025)  # Create a seeded random generator so results are repeatable.
resamples = rng.choice(data, size=(5000, n), replace=True)  # Resample WITH replacement: each pretend sample can reuse the same real reading, which mimics drawing a fresh sample. Repeat 5,000 times.
boot_means = resamples.mean(axis=1)  # Average each resample, giving 5,000 bootstrap means.

print('one resample:', resamples[0], ' mean =', resamples[0].mean())  # Print the first resample and its mean to see what a resample looks like.
print(f'average of boot means = {boot_means.mean():.2f}')  # Print the average of all bootstrap means; it should be close to the real mean.

one resample: [68 65 65 70 65]  mean = 66.6
average of boot means = 68.04


**What this means.** One resample was 68, 65, 65, 70, 65: the value 65 appears three times and 62 and 75 are missing, which is normal for sampling with replacement. Its mean is 66.6. Across 5,000 resamples the average mean is 68.04, close to the real 68.0.

## Step 3: Bootstrap SE and interval

The SD of the bootstrap means is the bootstrap standard error. The 2.5th and 97.5th percentiles of the means form a 95% percentile interval.

In [3]:
boot_se = boot_means.std(ddof=1)  # The SD of the 5,000 bootstrap means is the bootstrap standard error: how much a mean of 5 readings wobbles.
low, high = np.percentile(boot_means, [2.5, 97.5])  # Take the 2.5th and 97.5th percentiles of the bootstrap means: the middle 95% of them forms a 95% interval.

print(f'bootstrap SE = {boot_se:.2f}')  # Print the bootstrap standard error.
print(f'95% percentile interval = ({low:.1f}, {high:.1f})')  # Print the 95% percentile interval.

bootstrap SE = 1.99
95% percentile interval = (64.4, 72.0)


**What this means.** The bootstrap standard error is 1.99, close to the classic 2.21, so the two methods agree about how precise the mean is. The middle 95% of resampled means runs from 64.4 to 72.0 bpm.

## Step 4: Compare with the t-interval

With only five values the t-interval has to be wide to be honest. The bootstrap interval is narrower, because resampling cannot invent information a tiny sample does not contain.

In [4]:
t_low, t_high = stats.t.interval(0.95, df=n - 1, loc=data.mean(), scale=se)  # Compute the classic 95% t-interval for comparison; df = n - 1 and scale is the standard error.

print(f't-interval        = ({t_low:.1f}, {t_high:.1f})')  # Print the t-interval.
print(f'bootstrap interval = ({low:.1f}, {high:.1f})')  # Print the bootstrap interval from the last step.
print(f'widths: t = {t_high - t_low:.1f}, bootstrap = {high - low:.1f}')  # Print both widths (upper minus lower) so we can compare them directly.

t-interval        = (61.9, 74.1)
bootstrap interval = (64.4, 72.0)
widths: t = 12.3, bootstrap = 7.6


**What this means.** The t-interval is (61.9, 74.1), width 12.3, while the bootstrap interval is (64.4, 72.0), width 7.6. With only five values the t-interval is deliberately wide and honest. The bootstrap interval is narrower because resampling cannot add information; it tends to be too optimistic in tiny samples.

## So, what is the answer?

The mean heart rate is 68.0 bpm. Resampling gives a standard error of about 2.0 (the formula gives 2.21) and a 95% interval of 64.4 to 72.0, against the t-interval 61.9 to 74.1. Caution: with only five readings the bootstrap interval is too narrow to trust; the wider t-interval is the safer statement.

## Now you try

A commuter logged the minutes of eight trips to work. Travel times are skewed by the odd bad day, so she wants the typical trip measured by the median, which has no simple standard error formula. Use the bootstrap.

**Data:** [24, 31, 28, 45, 26, 33, 29, 52]

1. Find the sample median and mean.
2. Bootstrap the median with 5,000 resamples and report its standard error.
3. Report a 95% percentile interval for the median.
4. Compare with the bootstrap SE of the mean.

*Hint:* Use rng.choice(commute, size=(5000, 8), replace=True), then np.median(..., axis=1) and np.percentile(..., [2.5, 97.5]).

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
commute = np.array([24, 31, 28, 45, 26, 33, 29, 52])
rng = np.random.default_rng(4)
res = rng.choice(commute, size=(5000, len(commute)), replace=True)
meds = np.median(res, axis=1)
means = res.mean(axis=1)
print('sample median =', np.median(commute), ' sample mean =', commute.mean())
print(f'bootstrap SE of median = {meds.std(ddof=1):.2f}')
print(f'bootstrap SE of mean   = {means.std(ddof=1):.2f}')
print('95% interval for the median:', np.percentile(meds, [2.5, 97.5]))

sample median = 30.0  sample mean = 33.5
bootstrap SE of median = 3.91
bootstrap SE of mean   = 3.26
95% interval for the median: [26. 45.]


**Answer.** The sample median is 30.0 minutes and the mean is 33.5 (pulled up by the 45 and 52). The bootstrap SE of the median is 3.91, larger than the SE of the mean at 3.26. The 95% percentile interval for the median is 26 to 45 minutes. The median is less precise here, and its interval is wide, because only eight trips is very little data.

## The same question in R, SPSS and Stata

**R**

```r
hr <- c(62, 70, 68, 75, 65)  # Store the five heart rates.
set.seed(2025)  # Fix the random seed.
boot_means <- replicate(5000, mean(sample(hr, replace = TRUE)))  # Resample the data with replacement 5,000 times and keep the mean of each resample.
sd(boot_means)  # Print the SD of the bootstrap means (bootstrap SE).
quantile(boot_means, c(0.025, 0.975))  # Print the 2.5% and 97.5% quantiles, the percentile interval.
t.test(hr)$conf.int  # Print the classic t-interval for comparison.
```

How to read it: sd(boot_means) should be about 1.99 and the quantiles about 64.4 and 72. The t.test line prints 61.85 and 74.15. Bootstrap numbers are close to Python's, not identical, but the t-interval is exact.

**SPSS** (Analyze > Descriptive Statistics > Explore, then press the Bootstrap button and tick Perform bootstrapping (Number of samples 5000, Percentile))

```
DATA LIST FREE / hr.
BEGIN DATA
62 70 68 75 65
END DATA.
BOOTSTRAP
  /SAMPLING METHOD=SIMPLE
  /VARIABLES INPUT=hr
  /CRITERIA CILEVEL=95 CITYPE=PERCENTILE NSAMPLES=5000
  /MISSING USER=EXCLUDE.
EXAMINE VARIABLES=hr
  /PLOT=NONE
  /STATISTICS=DESCRIPTIVES.
```

How to read it: In the Descriptives output, the ordinary table shows a 95% Confidence Interval for Mean of about 61.9 to 74.1. The Bootstrap table shows the bootstrap Std. Error near 2.0 and a 95% interval near 64.4 to 72.0. Results are close to Python's.

**Stata**

```
clear
input hr
62
70
68
75
65
end
bootstrap mean = r(mean), reps(5000) seed(2025): summarize hr
estat bootstrap, percentile
ci means hr
```

How to read it: After estat bootstrap, look at the Std. err. (about 2.0) and the row labelled (P) for the percentile interval, about 64.4 to 72. The ci means table gives the t-interval, 61.85 to 74.15. Numbers are close to, not identical to, Python.